In [1]:
import os

base_path = r"./facedectection.v2i.coco2"  # Use raw string literals
print(f"Current directory contents: {os.listdir('.')}")
print(f"Dataset folder contents: {os.listdir(base_path)}")
print(f"Train folder contents: {os.listdir(os.path.join(base_path, 'train'))}")
print(f"Test folder contents: {os.listdir(os.path.join(base_path, 'test'))}")
print(f"Valid folder contents: {os.listdir(os.path.join(base_path, 'valid'))}")


Current directory contents: ['facedectection.v2i.coco2', 'facedectection.v2i.coco2.zip', 'model.ipynb', 'test.ipynb']
Dataset folder contents: ['README.roboflow.txt', 'test', 'train', 'valid']
Train folder contents: ['idcard', 'picture']
Test folder contents: ['idcard', 'picture', '_annotations.coco.json']
Valid folder contents: ['idcard', 'picture', '_annotations.coco.json']


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, Lambda, BatchNormalization
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
import matplotlib.pyplot as plt

# Function to load and preprocess images from directory
def load_images_from_directory(directory, target_size=(32, 32)):
    images = []
    for filename in os.listdir(directory):
        img_path = os.path.join(directory, filename)
        if filename.endswith('.jpg') or filename.endswith('.png'):
            img = load_img(img_path, target_size=target_size)
            img = img_to_array(img) / 255.0  # Normalize pixel values to [0, 1]
            images.append(img)
    return np.array(images)

# Load images for train, validation, and test sets
train_idcard_images = load_images_from_directory('facedectection.v2i.coco2/train/idcard')
train_picture_images = load_images_from_directory('facedectection.v2i.coco2/train/picture')

valid_idcard_images = load_images_from_directory('facedectection.v2i.coco2/valid/idcard')
valid_picture_images = load_images_from_directory('facedectection.v2i.coco2/valid/picture')

test_idcard_images = load_images_from_directory('facedectection.v2i.coco2/test/idcard')
test_picture_images = load_images_from_directory('facedectection.v2i.coco2/test/picture')


# Step 3: Create pairs and labels
def create_pairs(idcard_images, picture_images, labels):
    pairs = []
    pair_labels = []
    for i in range(len(idcard_images)):
        for j in range(len(picture_images)):
            pairs.append([idcard_images[i], picture_images[j]])
            pair_labels.append(labels[i])  # Assuming the labels are already binary for similarity
    return np.array(pairs), np.array(pair_labels)

# Create pairs and labels for training, validation, and testing
train_pairs, train_labels = create_pairs(train_idcard_images, train_picture_images, labels=[1]*len(train_idcard_images))  # Example: All pairs are similar
valid_pairs, valid_labels = create_pairs(valid_idcard_images, valid_picture_images, labels=[1]*len(valid_idcard_images))  # Same for validation
test_pairs, test_labels = create_pairs(test_idcard_images, test_picture_images, labels=[1]*len(test_idcard_images))  # Same for testing

# Step 4: Define the Base Model (Shared Network for feature extraction)
def create_base_model(input_shape):
    input_image = Input(shape=input_shape)
    x = Conv2D(64, (3, 3), activation='relu')(input_image)
    x = BatchNormalization()(x)
    x = Conv2D(128, (3, 3), activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(256, (3, 3), activation='relu')(x)
    x = BatchNormalization()(x)
    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu')(x)
    return Model(input_image, x)

# Step 5: Define the Euclidean Distance function for similarity measurement
def euclidean_distance(vectors):
    x, y = vectors
    sum_square = K.sum(K.square(x - y), axis=1, keepdims=True)
    return K.sqrt(K.maximum(sum_square, K.epsilon()))

# Step 6: Define the Siamese Network
def create_siamese_model(input_shape):
    input_image_1 = Input(shape=input_shape)
    input_image_2 = Input(shape=input_shape)
    base_model = create_base_model(input_shape)
    processed_1 = base_model(input_image_1)
    processed_2 = base_model(input_image_2)
    distance = Lambda(euclidean_distance)([processed_1, processed_2])
    model = Model(inputs=[input_image_1, input_image_2], outputs=distance)
    return model

# Step 7: Compile the Siamese Model
input_shape = (32, 32, 3)
siamese_model = create_siamese_model(input_shape)
siamese_model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0001), metrics=['accuracy'])

# Model summary
siamese_model.summary()

# Step 8: Train the Model
history = siamese_model.fit(
    [train_pairs[:, 0], train_pairs[:, 1]], 
    train_labels, 
    batch_size=32, 
    epochs=20, 
    validation_data=([valid_pairs[:, 0], valid_pairs[:, 1]], valid_labels)
)

# Step 9: Plot the Training History (Accuracy)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Step 10: Evaluate the Model on Test Data
test_loss, test_accuracy = siamese_model.evaluate(
    [test_pairs[:, 0], test_pairs[:, 1]], 
    test_labels
)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

# Step 11: Predict Similarity between Two Images
def predict_similarity(image1, image2):
    image1 = np.expand_dims(image1, axis=0)
    image2 = np.expand_dims(image2, axis=0)
    distance = siamese_model.predict([image1, image2])
    return distance

# Example prediction
distance = predict_similarity(test_pairs[0][0], test_pairs[0][1])
print(f"Predicted distance between images: {distance}")


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 256)       │ 89,111,168 │ input_layer[0][0… │
│ (Functional)        │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1)         │          0 │ functional[0][0], │
│                     │                   │            │ functional[1][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 89,111,168 (339.93 MB)

 Trainable params: 89,109,248 (339.92 MB)

 Non-trainable params: 1,920 (7.50 KB)

Epoch 1/20
7615/7615 ━━━━━━━━━━━━━━━━━━━━ 6981s 916ms/step - accuracy: 1.0000 - loss: 1.1921e-07 - val_accuracy: 1.0000 - val_loss: 1.1921e-07
Epoch 2/20
7615/7615 ━━━━━━━━━━━━━━━━━━━━ 6516s 856ms/step - accuracy: 1.0000 - loss: 1.1921e-07 - val_accuracy: 1.0000 - val_loss: 1.1921e-07
Epoch 3/20
7615/7615 ━━━━━━━━━━━━━━━━━━━━ 6453s 847ms/step - accuracy: 1.0000 - loss: 1.1921e-07 - val_accuracy: 1.0000 - val_loss: 1.1921e-07
Epoch 4/20
7615/7615 ━━━━━━━━━━━━━━━━━━━━ 0s 841ms/step - accuracy: 1.0000 - loss: 1.1921e-07